
###  Synthetic Health Data Generator (Full Notebook)

This notebook generates **synthetic healthcare datasets** for multiple customers:

-  DICOM images
-  Blood report PDFs
-  Wearable time-series CSVs

Run cells **top → bottom**. You can also run individual sections independently.


## 1. Imports

In [ ]:

import os
import uuid
import json
import random
import csv
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
from faker import Faker
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
import pydicom
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid, CTImageStorage, MRImageStorage

fake = Faker()
random.seed(42)
np.random.seed(42)


## 2. Customer Setup

In [ ]:

CUSTOMERS = [
    {"name": "Amara Patel", "age": 34, "gender": "F", "condition": "Hypertension"},
    {"name": "John Whitfield", "age": 58, "gender": "M", "condition": "Diabetes Type 2"},
    {"name": "Mei Lin", "age": 42, "gender": "F", "condition": "Anaemia"},
    {"name": "Carlos Rivera", "age": 27, "gender": "M", "condition": "Healthy"},
    {"name": "Fatima Al-Sayed", "age": 65, "gender": "F", "condition": "Osteoporosis"},
]

def make_customer_id(name: str) -> str:
    slug = name.lower().replace(" ", "_").replace("-", "")
    short = str(uuid.uuid5(uuid.NAMESPACE_DNS, name))[:8].upper()
    return f"CUST_{slug}_{short}"


## 3. Folder Structure

In [ ]:

def create_folder_structure(base_dir: Path, customer_id: str):
    root = base_dir / customer_id
    paths = {
        "root": root,
        "dicom": root / "DICOM",
        "blood": root / "BloodReports",
        "wearables": root / "Wearables",
        "genomics": root / "Genomics",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


## 4. DICOM Generator

In [ ]:

MODALITIES = ["CT", "MR", "XR"]

def generate_dicom(customer, out_dir, n_files=2):
    paths = []
    dob = datetime.now() - timedelta(days=customer["age"] * 365)

    for i in range(n_files):
        modality = random.choice(MODALITIES)
        sop_class = CTImageStorage if modality == "CT" else MRImageStorage

        file_meta = FileMetaDataset()
        file_meta.MediaStorageSOPInstanceUID = generate_uid()
        file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

        ds = Dataset()
        ds.file_meta = file_meta

        ds.PatientName = customer["name"]
        ds.PatientID = customer["customer_id"]
        ds.Modality = modality

        pixel_array = np.random.randint(0, 4096, (64, 64), dtype=np.uint16)
        ds.Rows, ds.Columns = 64, 64
        ds.PixelData = pixel_array.tobytes()

        fname = out_dir / f"{modality}_{i}.dcm"
        pydicom.dcmwrite(str(fname), ds)
        paths.append(fname)

    return paths


## 5. Blood Report Generator

In [ ]:

def generate_blood_report_pdf(customer, out_dir):
    fname = out_dir / f"BloodReport_{customer['customer_id']}.pdf"
    doc = SimpleDocTemplate(str(fname), pagesize=A4)

    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph(f"Patient: {customer['name']}", styles["Title"]))
    elements.append(Spacer(1, 12))

    data = [["Test", "Value"]]
    for test in ["Hb", "WBC", "Platelets"]:
        data.append([test, str(random.randint(1, 100))])

    table = Table(data)
    elements.append(table)

    doc.build(elements)
    return fname


## 6. Wearable Data Generator

In [ ]:

def generate_wearable_csv(customer, out_dir, days=7):
    fname = out_dir / f"Wearable_{customer['customer_id']}.csv"

    rows = []
    start = datetime.now() - timedelta(days=days)

    for d in range(days):
        for h in range(24):
            ts = start + timedelta(days=d, hours=h)
            rows.append({
                "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
                "heart_rate": random.randint(60, 100)
            })

    with open(fname, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    return fname


## 7. Run All Generators

In [ ]:

base_dir = Path("customers")
base_dir.mkdir(exist_ok=True)

for cust in CUSTOMERS:
    cust["customer_id"] = make_customer_id(cust["name"])
    paths = create_folder_structure(base_dir, cust["customer_id"])

    print(f"Processing {cust['name']}...")

    generate_dicom(cust, paths["dicom"])
    generate_blood_report_pdf(cust, paths["blood"])
    generate_wearable_csv(cust, paths["wearables"])

print("Done!")
